# Problem 1/2 최신 결과 노트북

이 노트북은 팀원이 빠르게 읽고 검토할 수 있도록 만든 **보고서형 실행 노트북**입니다.

핵심 메시지:

- Random-unitary의 x축 `k`와 Hamiltonian projection의 x축 `t`는 서로 다른 parameter입니다.
- 따라서 같은 x축 위치로 두 방법을 직접 비교하면 안 됩니다.
- cross-mechanism 비교는 `MMD`와 `Wasserstein-type distance`가 비슷한 지점을 matching해서 수행합니다.
- Problem 3는 최신 seed sweep 기준 `20/20` seeds가 `use_as_main`입니다.

읽는 순서:

1. **핵심 결론**을 먼저 읽습니다.
2. **김건우 팀원 이슈에 대한 답변**을 확인합니다.
3. 그림 2개와 comparable-strength table만 봐도 발표/보고서의 뼈대를 이해할 수 있습니다.
4. 마지막의 optional cell은 결과를 재생성할 때만 실행합니다.


## 1. 핵심 결론

| 항목 | 최신 값 | 해석 |
| --- | ---: | --- |
| Random-unitary final step | `k = 12` | Problem 1 scrambling baseline의 마지막 지점 |
| Random-unitary final MMD | `0.828093` | `S0`에서 충분히 멀어짐 |
| Random-unitary final Wasserstein | `0.686108` | infidelity-cost 기준 거리 |
| Hamiltonian max MMD | `1.249244` at `t=1.000000` | fixed Hamiltonian/projection의 최대 MMD |
| Hamiltonian max Wasserstein | `0.883968` at `t=4.000000` | fixed Hamiltonian/projection의 최대 W |
| MMD comparable pair | `k=1` vs `t=0.333333` | MMD gap `0.002259` |
| Wasserstein comparable pair | `k=7` vs `t=3.333333` | W gap `0.001220` |
| Problem 3 seed sweep | `20/20 use_as_main` | main result로 사용할 수 있는 robustness |
| Problem 3 MMD improvement | `0.097056` | seed sweep median |
| Problem 3 Wasserstein improvement | `0.147983` | seed sweep median |
| Axis-only score margin | `0.010000` | 작으므로 limitation에 명시 |


## 2. 김건우 팀원 이슈에 대한 답변

지적이 맞습니다. Random-unitary의 `k`는 discrete random circuit layer 수이고, Hamiltonian projection의 `t`는 continuous Hamiltonian evolution time입니다. 두 값은 물리적 의미와 단위가 다르므로 같은 x축에 놓고 `k=1`과 `t=1`을 직접 비교하면 잘못된 해석이 됩니다.

수정된 비교 방식:

1. native parameter curve는 mechanism별로 분리해서 봅니다.
2. cross-mechanism 비교는 같은 output metric, 즉 MMD/Wasserstein 값이 가까운 지점을 matching합니다.
3. resource/control 비교는 matching된 comparable-strength pair에서만 수행합니다.

보고서에서는 다음 표현이 안전합니다.

> `k`와 `t`는 같은 축이 아니므로 직접 비교하지 않았다. 대신 같은 initial ensemble `S0`와 같은 distance metric을 사용해 diffusion strength가 비슷한 지점을 matching하고, 그 지점에서 control/resource proxy를 비교했다.


In [ ]:
from pathlib import Path
import csv
import re

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path(r"C:\Coding\Hackathon6Quantum")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

P12_DIR = PROJECT_ROOT / "results" / "problem_1_2_baseline"
P3_SUMMARY = PROJECT_ROOT / "results" / "problem_3_seed_sweep" / "seed_sweep_summary.md"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem 1/2 result dir:", P12_DIR)
print("Problem 3 seed summary:", P3_SUMMARY)


In [ ]:
def read_csv_rows(path):
    with Path(path).open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def get_md_value(text, label):
    pattern = rf"^\s*(?:-\s*)?{re.escape(label)}\s*:\s*`([^`]+)`"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1) if match else ""


def show_png(path, title, figsize=(11, 5)):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    image = plt.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    plt.title(title)
    plt.show()


def print_rows_as_table(headers, rows):
    widths = [len(header) for header in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    print(" | ".join(header.ljust(widths[i]) for i, header in enumerate(headers)))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row)))

random_rows = read_csv_rows(P12_DIR / "random_unitary_metrics.csv")
ham_rows = read_csv_rows(P12_DIR / "hamiltonian_metrics.csv")
match_rows = read_csv_rows(P12_DIR / "comparable_strength_resource_matches.csv")
p3_text = P3_SUMMARY.read_text(encoding="utf-8")


## 3. Native parameter curve

이 그림은 mechanism별 진행을 확인하기 위한 그림입니다.

- 왼쪽: random-unitary의 `step k`
- 오른쪽: Hamiltonian projection의 `time t`

두 panel의 x축은 서로 다른 의미입니다. 이 그림은 “각 방법 내부의 변화”를 보는 용도입니다.


In [ ]:
show_png(
    P12_DIR / "distance_curves.png",
    "Native parameter curves: random step k and Hamiltonian time t are separate",
)


## 4. Metric-aligned comparison

이 그림이 두 mechanism을 비교할 때 더 중요한 그림입니다.

- 왼쪽은 각 결과를 `(MMD, Wasserstein)` metric space에 놓습니다.
- 오른쪽은 MMD 기준, Wasserstein 기준으로 가장 가까운 comparable-strength pair를 보여줍니다.

즉, x축 parameter를 억지로 맞추지 않고 **결과로 나타난 diffusion strength**를 기준으로 비교합니다.


In [ ]:
show_png(
    P12_DIR / "metric_aligned_comparison.png",
    "Metric-aligned comparison: compare by output distance, not by raw x-axis",
)


## 5. Comparable-strength table

Problem 2(d)의 resource/control comparison은 아래 두 pair를 기준으로 말하면 됩니다.


In [ ]:
table_rows = []
for row in match_rows:
    table_rows.append([
        row["matched_by"],
        f"k={row['random_step']}",
        f"t={float(row['hamiltonian_time']):.6f}",
        f"{float(row['mmd_gap']):.6f}",
        f"{float(row['wasserstein_gap']):.6f}",
        row["random_controls"],
        row["random_two_qubit_entanglers"],
        row["hamiltonian_fixed_terms"],
    ])

print_rows_as_table(
    ["matched_by", "random", "Hamiltonian", "MMD gap", "W gap", "random controls", "entanglers", "H terms"],
    table_rows,
)


## 6. Problem 3 seed sweep

Problem 3는 우리 팀의 차별화 포인트입니다. 아래 숫자는 최종 보고서/발표에서 main claim을 방어할 때 사용할 최신 robustness evidence입니다.


In [ ]:
p3_table = [
    ["recommendation", get_md_value(p3_text, "Main-claim recommendation")],
    ["total seeds", get_md_value(p3_text, "Total seeds")],
    ["use_as_main seeds", get_md_value(p3_text, "use_as_main")],
    ["main_candidate row fraction", get_md_value(p3_text, "main_candidate row fraction")],
    ["median MMD improvement", get_md_value(p3_text, "continuous_mmd_improvement")],
    ["median Wasserstein improvement", get_md_value(p3_text, "continuous_wasserstein_improvement")],
    ["axis-only score margin", get_md_value(p3_text, "continuous_score_minus_axis_score")],
    ["diversity retention", get_md_value(p3_text, "continuous_diversity_retention")],
    ["success probability", get_md_value(p3_text, "continuous_mean_success_probability")],
]
print_rows_as_table(["metric", "value"], p3_table)


## 7. 보고서에 넣을 수 있는 문장

Problem 1은 random circuit layer 수 `k`에 따른 scrambling baseline이고, Problem 2는 fixed Hamiltonian evolution time `t`와 complement projection에 따른 projected diffusion baseline이다. 두 parameter는 같은 단위가 아니므로 horizontal axis 위치를 직접 비교하지 않는다. 대신 같은 initial ensemble `S0`와 같은 fidelity-based MMD, infidelity-cost Wasserstein metric을 사용해 diffusion strength가 비슷한 지점을 matching하고, 그 지점에서 control/resource proxy를 비교한다.

최신 결과에서 MMD 기준 comparable pair는 random-unitary `k=1`과 Hamiltonian `t=0.333333`이며 MMD gap은 `0.002259`이다. Wasserstein 기준 comparable pair는 random-unitary `k=7`과 Hamiltonian `t=3.333333`이며 Wasserstein gap은 `0.001220`이다. 이 비교는 Hamiltonian 방법이 절대적으로 우월하다는 뜻이 아니라, random layer-wise control과 fixed Hamiltonian time/projection control이 서로 다른 비용 구조를 갖는다는 trade-off를 보여준다.

Problem 3 seed sweep는 `20/20 use_as_main`이며, median MMD improvement `0.097056`, median Wasserstein improvement `0.147983`을 보인다. 다만 axis-only score margin은 `0.010000`으로 작으므로, hardware advantage나 general quantum advantage가 아니라 small-scale post-selected toy proxy에서의 재현 가능한 개선으로 제한해 주장한다.


## 8. Optional: 결과 재생성

아래 cell은 결과를 새로 만들 때만 실행합니다. 이미 최신 결과가 있으면 실행하지 않아도 됩니다.


In [ ]:
RUN_REGENERATION = False

if RUN_REGENERATION:
    import subprocess
    subprocess.run(["python", "scripts/run_problem_1_2_baselines.py"], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["python", "scripts/run_quantitative_evaluation.py"], cwd=PROJECT_ROOT, check=True)
    print("Regenerated Problem 1/2 outputs and quantitative evaluation.")
else:
    print("Skipped regeneration. Set RUN_REGENERATION = True to refresh outputs.")
